## Projet - Prediction des ventes hebdomadaire

## Dataset Walmart sales forecast

### Objectif

Le projet vise à consevoir un système intelligent capable de prévoir les ventes hebdomadaires dans les magasins de distribution à partir des données historiques. Le système exploitera les données historiques des ventes, les patterns temporels et saisonniers.

Plan du notebook:

1. Préparation de l'environnement de travail
2. Chargement, fusion et inspection du dataset
3. Analyse Exploratoire
4. Feature Engineering
5. Modélisation et évaluation
7. Deploiement



### 1. Preparation de l'environnement de travail

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit,learning_curve
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
        Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten, BatchNormalization
    )
pd.set_option("display.max_columns",None)


### 2. Chargement, Inspection et Fusion des tables et fusion

### 2.1.  Chargement et inspection

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
store = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), str(1)
memory usage: 13.3 MB


In [4]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [5]:
#Affichons les differentes boutiques
print(f"Listons les boutiques enregistrées dans le fichier train")
train["Store"].unique()

Listons les boutiques enregistrées dans le fichier train


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [6]:
#Affichons les informations de la table store
print("Affichons les informations sommaires de la table store")
store.info()

Affichons les informations sommaires de la table store
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Store   45 non-null     int64
 1   Type    45 non-null     str  
 2   Size    45 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB


In [7]:
#Affichons les 5 premieres lignes
store.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [8]:
print("Affichons toutes les modalités disponible de la variable Store")
store["Store"].unique()

Affichons toutes les modalités disponible de la variable Store


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [9]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   str    
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), str(1)
memory usage: 712.0 KB


In [10]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


####  2.2.  Fusion des dataset

### A. Fusion du fichiers train, features et stores

In [11]:
df_train=pd.merge(train,
              store,
              how="left",
              on="Store",
                 indicator=True)
assert (df_train["_merge"]=="both").all()  #Cela nous permettra de verifier si tous les stores repertoriés dans train ont une correspondance dans Store
df_train=df_train.drop("_merge",axis=1) #Suppression de la colonne _merge après vérification
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size
0,1,1,2010-02-05,24924.50,False,A,151315
1,1,1,2010-02-12,46039.49,True,A,151315
2,1,1,2010-02-19,41595.55,False,A,151315
3,1,1,2010-02-26,19403.54,False,A,151315
4,1,1,2010-03-05,21827.90,False,A,151315


In [13]:
df_train=pd.merge(df_train,
                 features,
                 how="left",
                 on=["Store","Date","IsHoliday"],
                 indicator=True)
#Statisques de fusion
merge_stats=df_train["_merge"].value_counts()
print("Statistiques de fusion ")
print(merge_stats)
print(f"Ligne de la table Train ayant héritées des NaN après fusion {(df_train['_merge']=='Left_only').sum()} ")
df_train=df_train.drop("_merge",axis=1)   #Suppression de la colonne d'audit
#Vérifions si les Holiday_x et Holiday_y correspondent
if "IsHoliday_x" in df_train.columns and "IsHoliday_y" in df_train.columns:
    no_matching=(df_train["IsHoliday_x"]!=df_train["IsHoliday_y"])
    if no_matching.sum()>0:
        print(f"Attention: { no_matching.sum() } lignes de IsHoliday_x ne correspondent pas à IsHoliday_y")
    # Harmonisation
    df_train["IsHoliday"]=df_train["IsHoliday_x"]| df_train["IsHoliday_y"]
    df_train=df_train.drop(["IsHoliday_x","IsHoliday_y"],axis=1)


df_train.info()

Statistiques de fusion 
_merge
both          421570
left_only          0
right_only         0
Name: count, dtype: int64
Ligne de la table Train ayant héritées des NaN après fusion 0 
<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
 5   Type          421570 non-null  str    
 6   Size          421570 non-null  int64  
 7   Temperature   421570 non-null  float64
 8   Fuel_Price    421570 non-null  float64
 9   MarkDown1     150681 non-null  float64
 10  MarkDown2     111248 non-null  float64
 11  MarkDown3     137091 non-null  float64
 12  MarkDown4     134967 non-null  float64
 13  MarkDown5     151432 non-null  float64
 14  CPI     

B. Fusion du fichier test features et stores

In [14]:
test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [15]:
df_test=pd.merge(test,
              store,
              how="left",
              on="Store",
                 indicator=True)
assert (df_test["_merge"]=="both").all()  #Cela nous permettra de verifier si tous les stores repertoriés dans train ont une correspondance dans Store
df_test=df_test.drop("_merge",axis=1) #Suppression de la colonne _merge après vérification
df_test.head()

,Store,Dept,Date,IsHoliday,Type,Size
0,1,1,2012-11-02,False,A,151315
1,1,1,2012-11-09,False,A,151315
2,1,1,2012-11-16,False,A,151315
3,1,1,2012-11-23,True,A,151315
4,1,1,2012-11-30,False,A,151315


In [ ]:
df_test=pd.merge(df_test,
                 features,
                 how="left",
                 on=["Store","Date"],
                 indicator=True)
#Statisques de fusion
merge_stats=df_test["_merge"].value_counts()
print("Statistiques de fusion ")
print(merge_stats)
print(f"Ligne de la table Test ayant héritées des NaN après fusion {(df_test["_merge"]=="Left_only").sum()} ")
df_test=df_test.drop("_merge",axis=1)   #Suppression de la colonne d'audit
#Vérifions si les Holiday_x et Holiday_y correspondent
if "IsHoliday_x" in df_test.columns and "IsHoliday_y" in df_test.columns:
    no_matching=(df_test["IsHoliday_x"]!=df_test["IsHoliday_y"])
    if no_matching.sum()>0:
        print(f"Attention: { no_matching.sum() } lignes de IsHoliday_x ne correspondent pas à IsHoliday_y")
    # Harmonisation
    df_test["IsHoliday"]=df_test["IsHoliday_x"]| df_test["IsHoliday_y"]
    df_test=df_test.drop(["IsHoliday_x","IsHoliday_y"],axis=1)


df_test.head()

Statistiques de fusion 
_merge
both          115064
left_only          0
right_only         0
Name: count, dtype: int64
Ligne de la table Test ayant héritées des NaN après fusion 0 


,Store,Dept,Date,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,1,2012-11-02,A,151315,55.32,3.386,6766.44,5147.70,50.82,3639.90,2737.42,223.462779,6.573,False
1,1,1,2012-11-09,A,151315,61.24,3.314,11421.32,3370.89,40.28,4646.79,6154.16,223.481307,6.573,False
2,1,1,2012-11-16,A,151315,52.92,3.252,9696.28,292.10,103.78,1133.15,6612.69,223.512911,6.573,False
3,1,1,2012-11-23,A,151315,56.23,3.211,883.59,4.17,74910.32,209.91,303.32,223.561947,6.573,True
4,1,1,2012-11-30,A,151315,52.34,3.207,2460.03,NaN,3838.35,150.57,6966.34,223.610984,6.573,False


In [16]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 115064 entries, 0 to 115063
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   Store      115064 non-null  int64
 1   Dept       115064 non-null  int64
 2   Date       115064 non-null  str  
 3   IsHoliday  115064 non-null  bool 
 4   Type       115064 non-null  str  
 5   Size       115064 non-null  int64
dtypes: bool(1), int64(3), str(2)
memory usage: 4.5 MB


### 3. Exploration des données

### 3.1- Valeurs manquantes

In [17]:
print("Dataset informations")
print("*"*100)
print(f"Dimension du jeu d'entrainement: {df_train.shape}")
print(f"Dimension du jeu de test: {df_test.shape}")
print(f"Variables: {list(df_test.columns)}")
print("\n types de chacune des colonnes:")
print(df_train.dtypes)
print("*"*100)


Dataset informations
****************************************************************************************************
Dimension du jeu d'entrainement: (421570, 16)
Dimension du jeu de test: (115064, 6)
Variables: ['Store', 'Dept', 'Date', 'IsHoliday', 'Type', 'Size']

 types de chacune des colonnes:
Store             int64
Dept              int64
Date                str
Weekly_Sales    float64
IsHoliday          bool
Type                str
Size              int64
Temperature     float64
Fuel_Price      float64
MarkDown1       float64
MarkDown2       float64
MarkDown3       float64
MarkDown4       float64
MarkDown5       float64
CPI             float64
Unemployment    float64
dtype: object
****************************************************************************************************


In [18]:
#Définissons une fonction missing value qui nous permettra d'afficher les valeurs manquantes
def missing_values(df):
  """
  Cette fonction prendra en paramètre un dataframe et affichera les colonnes ayant de valeurs manquantes
  et le pourcentage correspondant.
  """
  missing_summary=pd.DataFrame({
    "Missing_count":df.isna().sum(),
    "Missing_percent":(df.isna().mean()*100).round(2)

})
  missing_summary=missing_summary[missing_summary["Missing_count"]>0].sort_values("Missing_count",ascending=False)
  print("Sommaire des valeurs manquantes")
  print(missing_summary)
missing_values(df_train)

Sommaire des valeurs manquantes
           Missing_count  Missing_percent
MarkDown2         310322            73.61
MarkDown4         286603            67.98
MarkDown3         284479            67.48
MarkDown1         270889            64.26
MarkDown5         270138            64.08


**Puisque les valeurs manquantes ne concernes que les colonnnes dediées aux promtions analysons ce processus de manquement en filtrant les semaines ayant au moins une promotion et celle n'ayant aucune promotion**

**Ce qu'il faut constater ici c'est que les NaN présent dans les differentes colonnes de promotions ne sont pas en réalité des valeurs manquantes mais plutôt le signe d'une absence de promotions pour cette semaine là**

### Nous allons remplacer ces NaN par des 0 mais avant cela nous allons essayer de créer des nouvelles colonnes qui permettrons d'analyser l'impact des promotions sur les ventes

In [20]:
markdown_cols=["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
df_train["Promotion_Active"]=(~df_train[markdown_cols].isnull().all(axis=1)).astype(int) #elle vaudra 1 si la semaine possède une promotion et 0 sinon
df_train["Nombre_Promotion"]=df_train[markdown_cols].notna().sum(axis=1)
#Determinons laquelles des promotions est réellement acctive
for col in markdown_cols:
    df_train[f"{col}_Active"]=df_train[col].notna().astype(int)
#Maintenant remplacons les NAN par 0
df_train[markdown_cols]=df_train[markdown_cols].fillna(0)
print("\n Profil des promotions par semaines: ")
print(f"Environ {df_train['Promotion_Active'].mean()*100:.1f}% des semaines possèdent au moins une promotion")


 Profil des promotions par semaines: 
Environ 35.9% des semaines possèdent au moins une promotion


In [21]:
df_train[df_train["Nombre_Promotion"]!=0] #combien de semaines sont influencées par les promotions

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active
92,1,1,2011-11-11,18689.54,False,A,151315,59.11,3.297,10382.90,6115.67,215.07,2406.62,6551.42,217.998085,7.866,1,5,1,1,1,1,1
93,1,1,2011-11-18,19050.66,False,A,151315,62.25,3.308,6074.12,254.39,51.98,427.39,5988.57,218.220509,7.866,1,5,1,1,1,1,1
94,1,1,2011-11-25,20911.25,True,A,151315,60.14,3.236,410.31,98.00,55805.51,8.00,554.92,218.467621,7.866,1,5,1,1,1,1,1
95,1,1,2011-12-02,25293.49,False,A,151315,48.91,3.172,5629.51,68.00,1398.11,2084.64,20475.32,218.714733,7.866,1,5,1,1,1,1,1
96,1,1,2011-12-09,33305.92,False,A,151315,43.93,3.158,4640.65,19.00,105.02,3639.42,14461.82,218.961846,7.866,1,5,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,False,B,118221,64.88,3.997,4556.61,20.64,1.50,1601.01,3288.25,192.013558,8.684,1,5,1,1,1,1,1
421566,45,98,2012-10-05,628.10,False,B,118221,64.89,3.985,5046.74,0.00,18.82,2253.43,2340.01,192.170412,8.667,1,4,1,0,1,1,1
421567,45,98,2012-10-12,1061.02,False,B,118221,54.47,4.000,1956.28,0.00,7.89,599.32,3990.54,192.327265,8.667,1,4,1,0,1,1,1
421568,45,98,2012-10-19,760.01,False,B,118221,56.47,3.969,2004.02,0.00,3.18,437.73,1537.49,192.330854,8.667,1,4,1,0,1,1,1


In [22]:
pd.set_option("display.max_columns",None)
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,0,0,0,0,0
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,0,0,0,0,0,0,0
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,0,0,0,0,0
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,0,0,0,0,0
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,0,0,0,0,0


### 3.2- Nombre de departement par store

In [24]:
print("*"*80)
print(f"Il existe très exactement { df_train['Dept'].nunique() } departements au sein de ce jeu de données")
print(f"Il existe très exactement { df_train['Store'].nunique() } stores au sein de ce jeu de données")
print("*"*80)

********************************************************************************
Il existe très exactement 81 departements au sein de ce jeu de données
Il existe très exactement 45 stores au sein de ce jeu de données
********************************************************************************


In [26]:
print("*"*60)
print(f"Explorons certaines colonnes plus en profondeur")
#Verifions s'il des valeurs négative ou nulles dans la variables Weekly_Sales
var_neg=(df_train["Weekly_Sales"]<=0).any() # Valeur booléen s'il au moins une valeur de Weekly_Sale est nulle ou négative
if var_neg: #si l'assertion est vrai affichons les lignes concernées
    print("La variable Weekly_Sales possèdes au moins une valeur negative ou nulle")
    print(f"Nombre de valeurs negatives ou nulles détectées: {df_train[df_train['Weekly_Sales']<=0].shape[0]}")
    Weekly_sales_neg=df_train[df_train['Weekly_Sales']<=0]
    print(f"Pourcentage de lignes donc la colonne Weekly_Sales est négative ou nulle: {(Weekly_sales_neg.shape[0]/df_train.shape[0])*100}%")
    print("*"*60)
    display(Weekly_sales_neg)

else:
    print("Toutes les valeurs de la variable WeeKly_Sales sont positives")

************************************************************
Explorons certaines colonnes plus en profondeur
La variable Weekly_Sales possèdes au moins une valeur negative ou nulle
Nombre de valeurs negatives ou nulles détectées: 1358
Pourcentage de lignes donc la colonne Weekly_Sales est négative ou nulle: 0.3221291837654482%
************************************************************


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active
846,1,6,2012-08-10,-139.65,False,A,151315,85.05,3.494,11436.22,245.0,6.85,6964.26,4836.22,221.958433,6.908,1,5,1,1,1,1,1
2384,1,18,2012-05-04,-1.27,False,A,151315,75.55,3.749,21290.13,0.0,69.89,4977.35,3261.04,221.671800,7.143,1,4,1,0,1,1,1
6048,1,47,2010-02-19,-863.00,False,A,151315,39.93,2.514,0.00,0.0,0.00,0.00,0.00,211.289143,8.106,0,0,0,0,0,0,0
6049,1,47,2010-03-12,-698.00,False,A,151315,57.79,2.667,0.00,0.0,0.00,0.00,0.00,211.380643,8.106,0,0,0,0,0,0,0
6051,1,47,2010-10-08,-58.00,False,A,151315,63.93,2.633,0.00,0.0,0.00,0.00,0.00,211.746754,7.838,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419597,45,80,2010-02-12,-0.43,True,B,118221,27.73,2.773,0.00,0.0,0.00,0.00,0.00,181.982317,8.992,0,0,0,0,0,0,0
419598,45,80,2010-02-19,-0.27,False,B,118221,31.27,2.745,0.00,0.0,0.00,0.00,0.00,182.034782,8.992,0,0,0,0,0,0,0
419603,45,80,2010-04-16,-1.61,False,B,118221,54.28,2.899,0.00,0.0,0.00,0.00,0.00,181.692477,8.899,0,0,0,0,0,0,0
419614,45,80,2010-07-02,-0.27,False,B,118221,76.61,2.815,0.00,0.0,0.00,0.00,0.00,182.318780,8.743,0,0,0,0,0,0,0


**Une semaine enregistrant des ventes negatives ou nulle serait certaine liée à un problème d'enregistrement car cela semble irréaliste d'enregistrer des ventes négatives ou nulle durant toute une semaine ou alors cela voudrait signifier qu'il y a eu plus de retour que de ventes durant cette semaine. Nous comptons 1358 lignes ayant ce potentiel problème. Puisque l'hypothèse selon laquelle ces valeur pourrait représenter une perte ou simplement une semaine sans vente nous allons créer une nouvelle variable binaire is_vente qui sera égale à 1 si la vente est strictement positive et 0 sinon**

### 3.3- Creation d'une variable Flag Is_Vente

In [27]:
df_train["Is_Vente"]=(~(df_train["Weekly_Sales"]<=0)).astype(int) #Cela nous permettra de savoir s'il y a effectivement vente au cours de la semaine
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active,Is_Vente
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,0,0,0,0,0,1
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,0,0,0,0,0,0,0,1
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,0,0,0,0,0,1
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,0,0,0,0,0,1
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,0,0,0,0,0,1


## 3.4- Examinons la colonne Date

In [28]:
print("*"*60)
print("affichons la date de debut de collecte des données et la date de fin ")
print(df_train['Date'].head(5))
df_train['Date'].tail()

************************************************************
affichons la date de debut de collecte des données et la date de fin 
0    2010-02-05
1    2010-02-12
2    2010-02-19
3    2010-02-26
4    2010-03-05
Name: Date, dtype: str


421565    2012-09-28
421566    2012-10-05
421567    2012-10-12
421568    2012-10-19
421569    2012-10-26
Name: Date, dtype: str

**Les données ont été collectées du 5 février 2010 au 26 octobre 2012 pour la suite de notre analyse nous allons convertir cette variable
en format datetime afin d'en créer de nouvelles colonnes**

In [31]:
print("Identifions les fêtes considérées dans notre jeu de données")
isholiday=df_train.loc[df_train["IsHoliday"]==True] #filtrons uniquement les lignes des semaines de fête
#Determinons toutes les differentes dates associées à cette fêtes
print(f"Nombre de semaines ordinaires: {df_train['Date'].nunique()-isholiday['Date'].nunique()}")
print(f"Nombre de semaine ayant une fête: {isholiday['Date'].nunique()}")
isholiday["Date"].unique()

Identifions les fêtes considérées dans notre jeu de données
Nombre de semaines ordinaires: 133
Nombre de semaine ayant une fête: 10


<StringArray>
['2010-02-12', '2010-09-10', '2010-11-26', '2010-12-31', '2011-02-11',
 '2011-09-09', '2011-11-25', '2011-12-30', '2012-02-10', '2012-09-07']
Length: 10, dtype: str

Il s'agit principalement :

* Du **Super Bowl** :12-Fev-2010, 11-Fev-2011 et 10-Fev-2012
* Du **Labor Day** 10-Sep-2010, 09-Sep-2011 et 07-Sep-2012
* De la fête de **Thanksgiving** 26-Nov-2010, 25-Nov-2011
* De la fête de **Noel** 31-Dec-2010 et 30-Dec-2011

Nous constatons entre autre l'année 2012 ne contient pas le dermier trimistre de l'année. De plus la semaine de noel est désignée comme étant la derniere de l'année.

Nous allons determiné la semaine de noel comme étant la 51 ieme de chaque année 

**Afin de mieux analyser impact de ces fêtes sur le chiffre d'affaire hebdomadaire nous allons créer une nouvelle colonne qui donnera laquelle des fêtes il s'agit en fonction de la date ainsi on saura laquelle influence le plus les ventes**

### 3.5- Creation des variables is_Thanksgiving, Is_Chrismas, Is_SuperBowl et Is_LaborDay

In [32]:
superbowl_date=["2010-02-12","2011-02-11","2012-02-10"]
labor_date=["2010-09-10","2011-09-09","2012-09-07"]
noel_date=["2010-12-24","2011-12-23"]
thank_date=["2010-11-26","2011-11-25"]
df_train["Is_SuperBowl"]=df_train["Date"].isin(superbowl_date).astype(int)
df_train["Is_LaborDay"]=df_train["Date"].isin(labor_date).astype(int)
df_train["Is_Chrismas"]=df_train["Date"].isin(noel_date).astype(int)
df_train["Is_Thankgiving"]=df_train["Date"].isin(thank_date).astype(int)
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active,Is_Vente,Is_SuperBowl,Is_LaborDay,Is_Chrismas,Is_Thankgiving
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,0,0,0,0,0,1,0,0,0,0
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,0,0,0,0,0,0,0,1,1,0,0,0
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,0,0,0,0,0,1,0,0,0,0
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,0,0,0,0,0,1,0,0,0,0
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,0,0,0,0,0,1,0,0,0,0


### 3.6- Création des variables temporelles et des variables

In [33]:
## Etape 1 convertissons la date en format datetime
df_train["Date"]=pd.to_datetime(df_train["Date"])
df_train.info()
## Etape 2 Création des variables years, Mounth et Week
df_train["Years"]=df_train["Date"].dt.year
df_train["Mois"]=df_train["Date"].dt.month_name()
df_train["Week"]=df_train["Date"].dt.isocalendar().week
df_train.head()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 28 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Store             421570 non-null  int64         
 1   Dept              421570 non-null  int64         
 2   Date              421570 non-null  datetime64[us]
 3   Weekly_Sales      421570 non-null  float64       
 4   IsHoliday         421570 non-null  bool          
 5   Type              421570 non-null  str           
 6   Size              421570 non-null  int64         
 7   Temperature       421570 non-null  float64       
 8   Fuel_Price        421570 non-null  float64       
 9   MarkDown1         421570 non-null  float64       
 10  MarkDown2         421570 non-null  float64       
 11  MarkDown3         421570 non-null  float64       
 12  MarkDown4         421570 non-null  float64       
 13  MarkDown5         421570 non-null  float64       
 14  CPI            

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active,Is_Vente,Is_SuperBowl,Is_LaborDay,Is_Chrismas,Is_Thankgiving,Years,Mois,Week
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,0,0,0,0,0,1,0,0,0,0,2010,February,5
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,0,0,0,0,0,0,0,1,1,0,0,0,2010,February,6
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,0,0,0,0,0,1,0,0,0,0,2010,February,7
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,0,0,0,0,0,1,0,0,0,0,2010,February,8
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,0,0,0,0,0,1,0,0,0,0,2010,March,9


In [34]:
df_noel=df_train.loc[df_train["Week"]==51,"Date"]
print(df_noel.unique())

<DatetimeArray>
['2010-12-24 00:00:00', '2011-12-23 00:00:00']
Length: 2, dtype: datetime64[us]


### 3.7- Inspection des markdows

In [35]:
has_negative = (df_train[markdown_cols] < 0).any().any()
neg_rows = df_train[(df_train[markdown_cols] < 0).any(axis=1)]
if has_negative:
    display(neg_rows[neg_rows["MarkDown2"]<0])

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Promotion_Active,Nombre_Promotion,MarkDown1_Active,MarkDown2_Active,MarkDown3_Active,MarkDown4_Active,MarkDown5_Active,Is_Vente,Is_SuperBowl,Is_LaborDay,Is_Chrismas,Is_Thankgiving,Years,Mois,Week
29629,4,1,2012-03-23,37148.64,False,A,205863,59.07,3.759,8806.80,-10.5,5.99,739.14,4396.97,130.896645,4.607,1,5,1,1,1,1,1,1,0,0,0,0,2012,March,12
29772,4,2,2012-03-23,94894.85,False,A,205863,59.07,3.759,8806.80,-10.5,5.99,739.14,4396.97,130.896645,4.607,1,5,1,1,1,1,1,1,0,0,0,0,2012,March,12
29915,4,3,2012-03-23,11416.47,False,A,205863,59.07,3.759,8806.80,-10.5,5.99,739.14,4396.97,130.896645,4.607,1,5,1,1,1,1,1,1,0,0,0,0,2012,March,12
30058,4,4,2012-03-23,53921.90,False,A,205863,59.07,3.759,8806.80,-10.5,5.99,739.14,4396.97,130.896645,4.607,1,5,1,1,1,1,1,1,0,0,0,0,2012,March,12
30201,4,5,2012-03-23,43779.18,False,A,205863,59.07,3.759,8806.80,-10.5,5.99,739.14,4396.97,130.896645,4.607,1,5,1,1,1,1,1,1,0,0,0,0,2012,March,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390864,41,97,2012-08-24,23337.75,False,A,196321,69.07,3.558,6068.27,-20.0,85.74,3710.05,2287.85,198.098420,6.432,1,5,1,1,1,1,1,1,0,0,0,0,2012,August,34
390995,41,98,2012-06-01,10433.00,False,A,196321,57.29,3.764,9160.25,-2.0,127.56,1850.68,2652.68,197.621895,6.547,1,5,1,1,1,1,1,1,0,0,0,0,2012,June,22
391007,41,98,2012-08-24,9822.45,False,A,196321,69.07,3.558,6068.27,-20.0,85.74,3710.05,2287.85,198.098420,6.432,1,5,1,1,1,1,1,1,0,0,0,0,2012,August,34
391041,41,99,2012-06-01,29.88,False,A,196321,57.29,3.764,9160.25,-2.0,127.56,1850.68,2652.68,197.621895,6.547,1,5,1,1,1,1,1,1,0,0,0,0,2012,June,22
